In [10]:
!nvidia-smi

Sun Jun 14 11:28:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P0             27W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [11]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.3/71.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.6/495.6 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 14.5 MB/s eta 0:00:00


In [12]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


In [13]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
from datasets import load_dataset
import pandas as pd
import evaluate
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import nltk
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import torch
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [14]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [15]:
model_ckpt = "google/pegasus-large"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

config.json:   0%|          | 0.00/3.09k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] Error during conversion: ReadTimeout('The read operation timed out')


model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-large
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

In [6]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
dataset_samsum = load_dataset("knkarthick/samsum")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/4.36k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/9.26M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/504k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/522k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

In [ ]:
print(dataset_samsum)

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})


In [7]:
def convert_examples_to_features(example_batch):

    model_inputs = tokenizer(
        example_batch["dialogue"],
        max_length=1024,
        truncation=True
    )

    labels = tokenizer(
        example_batch["summary"],
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [16]:
dataset_samsum_pt = dataset_samsum.map(
    convert_examples_to_features,
    batched=True
)

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

In [ ]:
dataset_samsum_pt["train"]

Dataset({
    features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 14731
})

In [ ]:
#Training
from transformers import DataCollatorForSeq2Seq
seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

In [ ]:
import transformers
print(transformers.__version__)

5.12.0


In [21]:
from transformers import TrainingArguments

trainer_args = TrainingArguments(
    output_dir="pegasus-samsum",

    num_train_epochs=1,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=16,

    fp16=True,                    # Huge memory savings

    warmup_steps=100,             # 500 is usually too high for SAMSum

    weight_decay=0.01,

    logging_steps=50,

    eval_strategy="steps",
    eval_steps=1000,

    save_strategy="epoch",

    dataloader_num_workers=2,

    report_to="none"
)

In [22]:
trainer = Trainer(
    model=model_pegasus,
    args=trainer_args,
    train_dataset=dataset_samsum_pt["train"].select(range(500)),
    eval_dataset=dataset_samsum_pt["validation"].select(range(100))
)

In [24]:
trainer.train()

Step,Training Loss,Validation Loss
32,No log,2.507775


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=32, training_loss=48.73812484741211, metrics={'train_runtime': 226.0667, 'train_samples_per_second': 2.212, 'train_steps_per_second': 0.142, 'total_flos': 189951246065664.0, 'train_loss': 48.73812484741211, 'epoch': 1.0})

In [25]:
from tqdm import tqdm
import torch

def generate_batch_sized_chunks(list_of_elements, batch_size):
    """Yield successive batch-sized chunks."""
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i:i + batch_size]


def calculate_metric_on_test_dataset(
    dataset,
    metric,
    model,
    tokenizer,
    batch_size=4,
    device=device,
    column_text="article",
    column_summary="highlights"
):

    article_batches = list(
        generate_batch_sized_chunks(dataset[column_text], batch_size)
    )

    target_batches = list(
        generate_batch_sized_chunks(dataset[column_summary], batch_size)
    )

    for article_batch, target_batch in tqdm(
        zip(article_batches, target_batches),
        total=len(article_batches)
    ):

        inputs = tokenizer(
            article_batch,
            max_length=512,  # reduced from 1024
            truncation=True,
            padding=True,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            summaries = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=128,
                num_beams=4
            )

        decoded_summaries = tokenizer.batch_decode(
            summaries,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )

        metric.add_batch(
            predictions=decoded_summaries,
            references=target_batch
        )

    score = metric.compute()
    return score

In [28]:
!pip install evaluate rouge_score

In [30]:
import evaluate

rouge_metric = evaluate.load("rouge")

In [40]:
score = calculate_metric_on_test_dataset(
    dataset_samsum["test"].select(range(10)),
    rouge_metric,
    trainer.model,
    tokenizer,
    batch_size=2,
    column_text="dialogue",
    column_summary="summary"
)

rouge_dict = {rn: score[rn] for rn in rouge_names}
pd.DataFrame(rouge_dict, index = [f'pegasus'])

100%|██████████| 5/5 [00:15<00:00,  3.07s/it]


,rouge1,rouge2,rougeL,rougeLsum
pegasus,0.249557,0.027624,0.174164,0.174754


In [41]:
print(score)

{'rouge1': np.float64(0.2495566330249688), 'rouge2': np.float64(0.02762360080706126), 'rougeL': np.float64(0.17416376832066327), 'rougeLsum': np.float64(0.1747535180477644)}


In [42]:
#save model
trainer.save_model("pegasus_samsum_model")
tokenizer.save_pretrained("pegasus_samsum_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('pegasus_samsum_model/tokenizer_config.json',
 'pegasus_samsum_model/tokenizer.json')

In [43]:
#save tokenizer
tokenizer.save_pretrained("tokenizer")

('tokenizer/tokenizer_config.json', 'tokenizer/tokenizer.json')

In [45]:
#load
tokenizer = AutoTokenizer.from_pretrained(
    "google/pegasus-cnn_dailymail"
)

config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

In [49]:
from transformers import PegasusForConditionalGeneration, PegasusTokenizer

model = PegasusForConditionalGeneration.from_pretrained(
    "./pegasus_samsum_model"
)

tokenizer = PegasusTokenizer.from_pretrained(
    "./tokenizer"
)

model.to(device)

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

PegasusForConditionalGeneration(
  (model): PegasusModel(
    (shared): Embedding(96103, 1024, padding_idx=0)
    (encoder): PegasusEncoder(
      (embed_tokens): Embedding(96103, 1024, padding_idx=0)
      (embed_positions): PegasusSinusoidalPositionalEmbedding(1024, 1024)
      (layers): ModuleList(
        (0-15): 16 x PegasusEncoderLayer(
          (self_attn): PegasusAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (final_layer_no

In [50]:
dialogue = dataset_samsum["test"][0]["dialogue"]

inputs = tokenizer(
    dialogue,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(device)

summary_ids = model.generate(
    **inputs,
    max_length=64,
    num_beams=8,
    early_stopping=True
)

prediction = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("Dialogue:\n", dialogue)
print("\nGenerated Summary:\n", prediction)
print("\nReference Summary:\n", dataset_samsum["test"][0]["summary"])

Dialogue:
 Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

Generated Summary:
 Amanda: Ask Larry Amanda: He called her last time we were at the park together Hannah: file_gif> Amanda: Don't be shy, he's very nice Hannah: If you say so..

Reference Summary:
 Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.


In [51]:
import transformers
print(transformers.__version__)

5.10.2


In [54]:
from transformers import PegasusForConditionalGeneration, PegasusTokenizer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = PegasusForConditionalGeneration.from_pretrained(
    "./pegasus_samsum_model"
).to(device)

tokenizer = PegasusTokenizer.from_pretrained("./tokenizer")

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

In [55]:
gen_kwargs = {
    "length_penalty": 0.8,
    "num_beams": 8,
    "max_length": 128
}

sample_text = dataset_samsum["test"][0]["dialogue"]
reference = dataset_samsum["test"][0]["summary"]

inputs = tokenizer(
    sample_text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    summary_ids = model.generate(
        **inputs,
        **gen_kwargs
    )

prediction = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("Dialogue:")
print(sample_text)

print("\nReference Summary:")
print(reference)

print("\nPredicted Summary:")
print(prediction)

Dialogue:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

Reference Summary:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.

Predicted Summary:
Amanda: Ask Larry Amanda: He called her last time we were at the park together Hannah: I don't know him well Hannah: file_gif> Amanda: Don't be shy, he's very nice Hannah: If you say so..


In [56]:
!ls

pegasus-samsum	pegasus_samsum_model  sample_data  tokenizer


In [57]:
import transformers
print(transformers.__version__)

5.10.2


In [58]:
!ls pegasus_samsum_model

config.json		model.safetensors      tokenizer.json
generation_config.json	tokenizer_config.json  training_args.bin


In [59]:
sample_text = dataset_samsum["test"][0]["dialogue"]

inputs = tokenizer(
    sample_text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

inputs = {k: v.to(device) for k, v in inputs.items()}

summary_ids = model.generate(
    **inputs,
    num_beams=8,
    max_length=128,
    length_penalty=0.8,
    early_stopping=True
)

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print(summary)

Amanda: Ask Larry Amanda: He called her last time we were at the park together Hannah: file_gif> Amanda: Don't be shy, he's very nice Hannah: If you say so..
